In [15]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, GlobalAveragePooling2D
import numpy as np
import matplotlib.pyplot as plt
import os
import cv2

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("marquis03/cats-and-dogs")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'cats-and-dogs' dataset.
Path to dataset files: /kaggle/input/cats-and-dogs


In [3]:
import shutil
import os

source_dir = path
destination_dir = "/content/Dataset1"

if os.path.exists(destination_dir):
    print(f"{destination_dir} already exists. Removing it to start fresh...")
    shutil.rmtree(destination_dir)

shutil.copytree(source_dir, destination_dir)
print(f"Dataset successfully copied from {source_dir} to {destination_dir}")

Dataset successfully copied from /kaggle/input/cats-and-dogs to /content/Dataset1


In [4]:
train_path = os.path.join("/content/Dataset1", "train")
test_path = os.path.join("/content/Dataset1", "val")

In [5]:
category = ['cat','dog']
category_info = {}
for cate in category:
  cate_path = os.path.join(train_path,cate)
  list_images = os.listdir(cate_path)
  print(f"{cate}:{len(list_images)}")
  category_info[cate] = [len(list_images),cate_path]

cat:95
dog:180


In [6]:
def DataAugmentation(cate_info):
    #Counting max image count
    max_count = max(c[0] for c in cate_info.values())
    #Augmentation image generator
    datagen = ImageDataGenerator(
        rotation_range=30,
        zoom_range=0.2,
        horizontal_flip=True,
        width_shift_range=0.2,
        height_shift_range=0.2,
        brightness_range=[0.7, 1.3],
        fill_mode="nearest")
    #loop for max category
    for cate in category:
      count, cate_path = cate_info[cate]
      required = max_count - count
      if required > 0:
        images = []
        #loop for reading all images
        for file in os.listdir(cate_path):
          img = cv2.imread(os.path.join(cate_path, file))
          if img is not None:
            img = cv2.resize(img, (224,224))
            images.append(img)
        #Generator for image generation
        images = np.array(images)
        generator = datagen.flow(
            images,
            batch_size = 1,
            save_to_dir = cate_path,
            save_prefix = "aug",
            save_format = 'jpg'
        )
        #loop for generating required images
        for i in range(required):
          next(generator)
          print(f"Image {i+1} Generated for category:{cate}")
DataAugmentation(cate_info=category_info)

Image 1 Generated for category:cat
Image 2 Generated for category:cat
Image 3 Generated for category:cat
Image 4 Generated for category:cat
Image 5 Generated for category:cat
Image 6 Generated for category:cat
Image 7 Generated for category:cat
Image 8 Generated for category:cat
Image 9 Generated for category:cat
Image 10 Generated for category:cat
Image 11 Generated for category:cat
Image 12 Generated for category:cat
Image 13 Generated for category:cat
Image 14 Generated for category:cat
Image 15 Generated for category:cat
Image 16 Generated for category:cat
Image 17 Generated for category:cat
Image 18 Generated for category:cat
Image 19 Generated for category:cat
Image 20 Generated for category:cat
Image 21 Generated for category:cat
Image 22 Generated for category:cat
Image 23 Generated for category:cat
Image 24 Generated for category:cat
Image 25 Generated for category:cat
Image 26 Generated for category:cat
Image 27 Generated for category:cat
Image 28 Generated for category:cat
I

In [26]:
train_gen = ImageDataGenerator(
    preprocessing_function = tf.keras.applications.vgg16.preprocess_input,
)
val_gen = ImageDataGenerator(
    preprocessing_function = tf.keras.applications.vgg16.preprocess_input,
)

In [27]:
train_data = train_gen.flow_from_directory(
    train_path,
    target_size=(224,224),
    batch_size=16,
    class_mode='binary'
)
val_data = val_gen.flow_from_directory(
    test_path,
    target_size=(224,224),
    batch_size=16,
    class_mode='binary'
)

Found 360 images belonging to 2 classes.
Found 70 images belonging to 2 classes.


In [28]:
base_model = tf.keras.applications.VGG16(
    include_top = False,
    input_shape = (224,224,3)
)
base_model.trainable = False

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [29]:

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dense(1, activation='sigmoid')
])

In [30]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,780,481 (56.38 MB)

 Trainable params: 65,793 (257.00 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [31]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy'])

In [32]:
history = model.fit(
    train_data,
    epochs=50,
    batch_size =16,
    verbose = 1,
    validation_data=val_data,
)

Epoch 1/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 33s 951ms/step - accuracy: 0.8944 - loss: 0.4198 - val_accuracy: 0.9429 - val_loss: 0.3143
Epoch 2/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 87ms/step - accuracy: 1.0000 - loss: 0.0051 - val_accuracy: 0.9714 - val_loss: 0.2389
Epoch 3/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 85ms/step - accuracy: 0.9972 - loss: 0.0046 - val_accuracy: 0.9714 - val_loss: 0.2089
Epoch 4/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 86ms/step - accuracy: 1.0000 - loss: 2.9130e-04 - val_accuracy: 0.9714 - val_loss: 0.1652
Epoch 5/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 86ms/step - accuracy: 1.0000 - loss: 2.8694e-04 - val_accuracy: 0.9714 - val_loss: 0.1760
Epoch 6/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 99ms/step - accuracy: 1.0000 - loss: 2.1833e-04 - val_accuracy: 0.9714 - val_loss: 0.1955
Epoch 7/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 91ms/step - accuracy: 1.0000 - loss: 1.7108e-04 - val_accuracy: 0.9714 - val_loss: 0.1915
Epoch 8/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 86ms/step - accuracy: 1.0000 - loss: 1.5823e-04 - va

In [33]:
def predict_image(img):
  image = cv2.imread(img)
  image = cv2.resize(image, (224,224))
  image = np.expand_dims(image, axis=0)
  image = tf.keras.applications.resnet.preprocess_input(image)
  pred = model.predict(image)
  return pred

In [34]:
pred_val = predict_image("/content/dog2.jpg")
if pred_val >= 0.5:
  print("dog")
else:
  print("cat")

1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step
dog
